# Champions League winner probabilities

Train a small neural classifier on *pre-tournament* team/season metrics. Each row is a team entering a season; `won` is 1 only for that season's champion. Keep every feature available before the competition begins.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
import tensorflow as tf

# Place Kaggle downloads here. Examples: UCL historical winners + football club/season statistics.
stats = pd.read_csv(Path('data/team_season_stats.csv'))
winners = pd.read_csv(Path('data/ucl_winners.csv'))
stats.head()

In [ ]:
# Adapt these mappings once to the chosen Kaggle files; do not merge in post-final stats.
stats = stats.rename(columns={'Season':'season', 'Team':'team'})
winners = winners.rename(columns={'Season':'season', 'Winner':'team'})
champions = winners[['season', 'team']].drop_duplicates().assign(won=1)
df = stats.merge(champions, on=['season','team'], how='left').assign(won=lambda x: x.won.fillna(0).astype(int))
features = df.select_dtypes(include='number').columns.drop('won', errors='ignore')
df = df.dropna(subset=['season']).sort_values('season')
features.tolist()

In [ ]:
# Chronological split: no information from later tournaments in validation.
cutoff = sorted(df.season.unique())[-5]
train, valid = df[df.season < cutoff], df[df.season >= cutoff]
scaler = StandardScaler()
X_train = scaler.fit_transform(train[features].fillna(train[features].median()))
X_valid = scaler.transform(valid[features].fillna(train[features].median()))
net = tf.keras.Sequential([tf.keras.layers.Input((X_train.shape[1],)), tf.keras.layers.Dense(32, activation='relu'), tf.keras.layers.Dropout(.25), tf.keras.layers.Dense(1, activation='sigmoid')])
net.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
net.fit(X_train, train.won, validation_data=(X_valid, valid.won), epochs=50, batch_size=32, callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)], verbose=0)
p = net.predict(X_valid, verbose=0).ravel()
print({'AUC': round(roc_auc_score(valid.won,p),3), 'Brier': round(brier_score_loss(valid.won,p),3)})

In [ ]:
# Normalize model scores within a season so they read as a winner probability.
forecast = valid[['season','team']].copy(); forecast['score'] = p
forecast['winner_probability'] = forecast.groupby('season').score.transform(lambda s: s / s.sum())
forecast.sort_values(['season','winner_probability'], ascending=[False,False]).groupby('season').head(5)